# पाठ 09 - मेटाकग्निसन डिजाइन ढाँचा


## सेटअप

यो नोटबुकले Microsoft Agent Framework प्रयोग गरेर Metacognition डिजाइन ढाँचा प्रदर्शन गर्दछ।

**पूर्वआवश्यकताहरू:**
- Azure OpenAI परिनियोजन वातावरण चरहरू मार्फत कन्फिगर गरिएको
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मेटासंज्ञान के हो?

मेटासंज्ञान भनेको **सोचबारेको सोच** हो। एआई एजेन्टहरूको सन्दर्भमा, यसको अर्थ त्यस्ता एजेन्टहरू निर्माण गर्नु हो जसले सकुन्:

- **आत्म-निरीक्षण** आफ्ना नतिजाहरू र तर्क प्रक्रियामा
- **त्रुटि पत्ता लगाउने** र चुपचाप असफल हुनेको सट्टा सुगमतापूर्वक पुन:उद्धार गर्ने
- **मूल्याङ्कन गर्ने** कि तिनीहरूको उत्तरहरू पूर्ण र उपयोगी छन् कि छैनन्
- **अनुकूलन गर्ने** जब प्रारम्भिक दृष्टिकोण काम गर्दैन (जस्तै, ब्याकअप प्रणालीमा फर्किनु)

एक मेटासंज्ञानिक एजेन्ट केवल प्रश्नहरूको उत्तर मात्र दिन्छ भन्ने होइन — यो आफ्नो प्रदर्शन अनुगमन गर्छ र तत्कालै समायोजन गर्छ।


## प्राथमिक र ब्याकअप उपकरणहरू

एक सामान्य मेटाकग्निशन ढाँचा **फलब्याक रणनीति** हो। एजेन्टले पहिले प्राथमिक उपकरण प्रयास गर्छ; यदि यो असफल भयो (उदाहरणका लागि, 404 त्रुटि), एजेन्टले असफलता पहिचान गर्छ र पारदर्शी रूपमा ब्याकअप उपकरणमा स्विच गर्छ।

यो वास्तविक-विश्व प्रणालीलाई प्रतिबिम्बित गर्छ जहाँ प्राथमिक सेवाहरू अनुपलब्ध हुन सक्छन् र एजेन्टहरूले वैकल्पिक मार्ग छान्नु अघि आफैंले समस्यालाई पत्ता लगाउनुपर्छ।

तल हामी दुई उडान खोज्ने उपकरणहरू परिभाषित गर्छौं:
- **प्राथमिक** — पेरिस, टोकियो, र बार्सिलोना समेट्छ
- **ब्याकअप** — बर्लिन, सिड्नी, र न्यूयोर्क सिटी समेट्छ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटि पुनर्प्राप्ति सहितको आत्म-प्रतिबिम्ब गर्ने एजेन्ट

तलको एजेन्टलाई पहिलोमा प्राथमिक उडान प्रणाली प्रयास गर्न, असफलताहरू चिन्हित गर्न, र पारदर्शी रूपमा ब्याकअप प्रणालीमा फर्कन निर्देशन दिइएको छ। प्रत्येक प्रतिक्रियापछि यसले संक्षेपमा आत्म-मूल्यांकन गर्छ कि यसले प्रयोगकर्ताको प्रश्नलाई पूर्ण रूपमा उत्तर दिएको छ कि छैन।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## आत्म-मूल्यांकन ढाँचा

मेटाकोग्निशनको अर्को पाटो **आत्म-मूल्यांकन** हो: एक अलग एजेन्ट (वा दोस्रो पासमा उही एजेन्ट) ले उत्तरको पूर्णता, सटिकता, र उपयोगिताका लागि समीक्षा गर्छ। तल हामी एउटा `ResponseEvaluator` एजेन्ट सिर्जना गर्छौं जसले यात्रा-एजेन्ट प्रतिक्रियाहरूलाई तीन आयाममा स्कोर गर्छ।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरेर **मेटाकग्निटिभ एजेन्टहरू** कसरी निर्माण गर्ने जान्नुभयो:

- **आत्म-प्रतिबिम्ब**: आफ्नै तर्कलाई अनुगमन गर्ने र पारदर्शी रूपमा के भयो भनेर सञ्चार गर्ने एजेन्टहरू।
- **त्रुटि पुनर्प्राप्ति र फलब्याकहरू**: एक प्रमुख + ब्याकअप उपकरण ढाँचा जहाँ एजेन्टले असफलता (जस्तै, 404 त्रुटिहरू) पत्ता लगाउँछ र स्वचालित रूपमा वैकल्पिक स्रोत प्रयास गर्छ।
- **आत्म-मूल्याङ्कन**: एउटा अलग मूल्याङ्कन गर्ने एजेन्ट जसले जवाफहरूलाई पूर्णता, सटीकता, र उपयोगिताका लागि गुणाङ्कन गर्छ।

यी ढाँचाहरूले एजेन्टहरूलाई थप बलियो, पारदर्शी, र विश्वसनीय बनाउँछन् — उत्पादन परिनियोजनहरूको लागि महत्वपूर्ण गुणहरू।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी यथासम्भव शुद्धता सुनिश्चित गर्न प्रयास गर्छौं, तर स्वचालित अनुवादहरूमा त्रुटि वा अशुद्धता हुनसक्छ भन्ने कुरामा कृपया सचेत रहनुहोस्। मूल दस्तावेजलाई त्यसको मूल भाषामा नै प्रामाणिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिश गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
